In [1]:
#to avoid Intel Jupyter kernel crash 
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import plotly.express as px
import torch 
import torch.nn as nn
import json
import math

from models import simpleLSTM
from utils import run_closed_loop, perf_measure

import matplotlib.dates as mdates
import matplotlib.pyplot as plt

SMALL_SIZE = 12
MEDIUM_SIZE = 12
BIGGER_SIZE = 12

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title


myFmt_year = mdates.DateFormatter('%Y-%m-%d')
myFmt_month_day_time = mdates.DateFormatter('%m-%d-%H:%M')
myFmt_month_day = mdates.DateFormatter('%m-%d')
myFmt = mdates.DateFormatter('%H:%M')


### RMSE of fifth Input

In [2]:
df_path = "./saved_runs/with_combined_pv_sizes/train_history.csv"
df = pd.read_csv(df_path)
fig = px.line(df, x="epoch", y="train_loss")
#fig.write_image("./plots/train_loss.pdf")  
fig.show()

In [3]:
model_path = "./saved_runs/with_combined_pv_sizes/best_valid_model.pt"
scaling_path = "./saved_runs/with_combined_pv_sizes/scalings.json"

#initializing the model
model_combined = simpleLSTM(n_input_features=5, num_layers=4) #default n_input_features=4
model_combined.load_state_dict(torch.load(model_path,map_location=torch.device("cpu")))

#getting the scaling of first model
with open(scaling_path, 'r') as file:
    scaling_dict = json.load(file)

#initializing the test data of the model you want to compare the first to
experiment_path_to_compare = "./saved_runs/with_pv0.4"
#getting the scalings of the second test set
scaling_data_path_to_compare = os.path.join(experiment_path_to_compare,'scalings.json')
with open(scaling_data_path_to_compare, 'r') as file:
    scaling_dict_to_compare = json.load(file)


max_power = scaling_dict_to_compare['NP']['max']
min_power = scaling_dict_to_compare['NP']['min']


#loading the test data
df_test_scaled_list = []
for i in range(6): #number of months
    test_data_path = os.path.join(experiment_path_to_compare, f'test{i}_data_scaled.csv')
    df_test_scaled = pd.read_csv(test_data_path, index_col = 0)
    # Convert timestamps to datetime objects 
    datetime_index = pd.to_datetime(df_test_scaled.index)
    df_test_scaled.index = datetime_index
    df_test_scaled_list.append(df_test_scaled)

    
lookback = 96 #24 hours

test_values = df_test_scaled_list[0].values #index 0 for first month (july)
scaling_dict_testset = test_values[0]
num_test_samples = test_values.shape[0]
steps2predict = num_test_samples-lookback

groundtruth = test_values[lookback+1::, 0] # There is a + 1 shift in the inference loop
#groundtruth = groundtruth * (max_power - min_power) + min_power
print("Groudtruth", groundtruth)
print(groundtruth.shape)
time = pd.Series(df_test_scaled_list[0].index)[lookback:-1]

predictions_with_pv = run_closed_loop(model_combined, test_values, lookback = lookback, future_prediction= steps2predict, use_positional_encoding="pv_const")
#predictions_with_pv = predictions_with_pv * (max_power - min_power) + min_power
print("predictions:",predictions_with_pv)
print(predictions_with_pv.shape)


# start time
start_time = "2021-07-15 00:00"
freq = "15min"

# index starts at end of lookback window
index_start = 96  
date_index = pd.date_range(start=start_time, periods=len(predictions_with_pv) + index_start, freq=freq)
prediction_index = date_index[index_start:]


df = pd.DataFrame(predictions_with_pv, columns=["Prediction"], index=prediction_index)
df["Groundtruth"] = groundtruth

fig = px.line(df, x=df.index, y=["Prediction", "Groundtruth"], title="Predicted Values of combined set with pv 0.4")
fig.update_layout(
    xaxis_title="Time",
    yaxis_title="Predicted Value",
    template="plotly_white",
    hovermode="x unified"
)
fig.show()
#predictions_with_combined_pv = run_closed_loop()


C:\Users\kivi-\AppData\Local\Temp\ipykernel_1808\3580865647.py:6: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



Groudtruth [0.61782822 0.61714258 0.60386691 ... 0.63634419 0.63366457 0.61820253]
(1343,)
predictions: [0.61208016 0.60396147 0.5974791  ... 0.6197217  0.61310107 0.60657924]
(1343,)


In [4]:
month_labels = ["July", "August", "September", "October", "November", "December"]
for month_i in range(6):
    test_values = df_test_scaled_list[month_i].values
    index = df_test_scaled_list[month_i].index

    num_test_samples = test_values.shape[0]
    steps2predict = num_test_samples - lookback

    # groundtruth rescaling
    gt = test_values[lookback+1::, 0]
    #gt = gt * (max_power - min_power) + min_power

    # prediction rescaling
    preds = run_closed_loop(model_combined, test_values, lookback=lookback, future_prediction=steps2predict, use_positional_encoding="pv_const")
    #preds = preds * (max_power - min_power) + min_power

    # time index 
    time_index = index[lookback:-1]

    # dataframe for plot
    df_month = pd.DataFrame({
        "Prediction with combined model": preds,
        "Groundtruth of 0.4 set": gt
    }, index=pd.to_datetime(time_index))

    # plotting
    fig = px.line(df_month, x=df_month.index, y=["Prediction with combined model", "Groundtruth of 0.4 set"], title=f"{month_labels[month_i]} – Predictions of combined model (0.2,0.6,1.0) vs Groundtruth of 0.4 Testset")
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title="Power normalised",
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

In [5]:
model_path04 = "./saved_runs/with_pv0.4/best_valid_model.pt"
model_pv04 = simpleLSTM(n_input_features=5, num_layers=4) #default n_input_features=4
model_pv04.load_state_dict(torch.load(model_path04,map_location=torch.device("cpu")))

models = {
    "Combined model": model_combined,
    "PV 0.4 model": model_pv04
}

rmse_results = {
    "Month": [],
    "Model": [],
    "Datetime": [],
    "time_step": [],
    "RMSE": []
}
month_labels = ["July", "August", "September", "October", "November", "December"]
lookback = 96

for model_name, model in models.items():
    global_time_idx = 0
    for month_i in range(6):
        test_values = df_test_scaled_list[month_i].values
        index = df_test_scaled_list[month_i].index
        time_index = index[lookback+1:]  

        groundtruth = test_values[lookback+1:, 0]
        #groundtruth = groundtruth * (max_power - min_power) + min_power

        preds = run_closed_loop(model, test_values, lookback=lookback, future_prediction=steps2predict, use_positional_encoding="pv_const")
        #preds = preds * (max_power - min_power) + min_power

       
        # RMSE 
        for i in range(len(groundtruth)):  
            rmse = np.sqrt((groundtruth[i] - preds[i]) ** 2)  

            rmse_results["Month"].append(month_labels[month_i])  # Month
            rmse_results["Model"].append(model_name)  # modelname
            rmse_results["Datetime"].append(time_index[i])  # datetime
            rmse_results["RMSE"].append(rmse)  # RMSE value
            rmse_results["time_step"].append(global_time_idx)
            global_time_idx +=1

rmse_df = pd.DataFrame(rmse_results)
print(rmse_df)
fig = px.line(
    rmse_df,
    x="Datetime",
    y="RMSE",
    color="Model",
    markers=True,
    title="RMSE Comparison: combined model vs. pv 0.4"
)

fig.update_layout(
    xaxis=dict(
        type='category'),
    xaxis_title="Month",
    yaxis_title="RMSE",
    template="plotly_white",
    hovermode="x unified"
)
fig.update_xaxes(tickmode='array', tickvals=rmse_df["Datetime"][::(96*14)],ticktext=[ts.strftime("%Y-%m") for ts in rmse_df["Datetime"][::(96*14)]])
fig.update_xaxes(rangeslider_visible=True)
fig.show()

C:\Users\kivi-\AppData\Local\Temp\ipykernel_1808\2257346886.py:3: FutureWarning:

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.



          Month           Model                  Datetime  time_step      RMSE
0          July  Combined model 2021-07-16 00:15:00+00:00          0  0.005748
1          July  Combined model 2021-07-16 00:30:00+00:00          1  0.013181
2          July  Combined model 2021-07-16 00:45:00+00:00          2  0.006388
3          July  Combined model 2021-07-16 01:00:00+00:00          3  0.023573
4          July  Combined model 2021-07-16 01:15:00+00:00          4  0.027661
...         ...             ...                       ...        ...       ...
16111  December    PV 0.4 model 2021-12-29 22:45:00+00:00       8053  0.040952
16112  December    PV 0.4 model 2021-12-29 23:00:00+00:00       8054  0.075894
16113  December    PV 0.4 model 2021-12-29 23:15:00+00:00       8055  0.073370
16114  December    PV 0.4 model 2021-12-29 23:30:00+00:00       8056  0.067415
16115  December    PV 0.4 model 2021-12-29 23:45:00+00:00       8057  0.053370

[16116 rows x 5 columns]
